In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
import os
os.chdir('/Users/lior/DS6021/DS6021_F26/data')
peng = pd.read_csv("penguins.csv").dropna(
    subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]
).reset_index(drop=True)
bike = pd.read_csv("bike-sharing-daily.csv", parse_dates=["dteday"])

print(peng.shape, bike.shape)

(333, 8) (731, 16)


In [2]:
#q1

sm_fit = smf.ols("body_mass_g ~ flipper_length_mm + C(sex) + C(species)", data=peng).fit()
print(sm_fit.summary().tables[1])
print(f"R^2 = {sm_fit.rsquared:.4f}")
print("sex levels:    ", sorted(peng.sex.unique()))
print("species levels:", sorted(peng.species.unique()))

                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                -365.8174    532.050     -0.688      0.492   -1412.479     680.844
C(sex)[T.male]            530.3811     37.810     14.027      0.000     456.000     604.762
C(species)[T.Chinstrap]   -87.6345     46.347     -1.891      0.060    -178.810       3.541
C(species)[T.Gentoo]      836.2600     85.185      9.817      0.000     668.681    1003.839
flipper_length_mm          20.0249      2.846      7.037      0.000      14.427      25.623
R^2 = 0.8669
sex levels:     ['female', 'male']
species levels: ['Adelie', 'Chinstrap', 'Gentoo']


intercept = −365.82, flipper_length_mm = 20.02, sex[male] = 530.38, species[Chinstrap] = −87.63, species[Gentoo] = 836.26. R² = 0.8669.

The reference level for sex is female and Adelie for species. 

C(species)[T.Gentoo]= 836.26 and the when flipper length and sex is fixed, the gentoo weighs 836.26 grams more than the Adelie.



In [3]:
#q2- a

y =peng["body_mass_g"]
X = pd.get_dummies(peng[["flipper_length_mm", "sex", "species"]], drop_first=True, dtype=float)
print(list(X.columns))

lr =LinearRegression().fit(X, y)
print("intercept_:",round(lr.intercept_, 4))
print(pd.Series(lr.coef_, index=X.columns).round(4))
print("R^2:", round(lr.score(X, y), 4))
print("\nstatsmodels params:"); print(sm_fit.params.round(4))
print("Same R^2:", np.isclose(lr.score(X, y), sm_fit.rsquared))
print("Same predictions: ", np.allclose(lr.predict(X), sm_fit.fittedvalues))

['flipper_length_mm', 'sex_male', 'species_Chinstrap', 'species_Gentoo']
intercept_: -365.8174
flipper_length_mm     20.0249
sex_male             530.3811
species_Chinstrap    -87.6345
species_Gentoo       836.2600
dtype: float64
R^2: 0.8669

statsmodels params:
Intercept                 -365.8174
C(sex)[T.male]             530.3811
C(species)[T.Chinstrap]    -87.6345
C(species)[T.Gentoo]       836.2600
flipper_length_mm           20.0249
dtype: float64
Same R^2: True
Same predictions:  True


In [4]:
#2b

rng = np.random.default_rng(42)
rows = []
for _ in range(1000):
    s = peng.sample(len(peng),replace=True, random_state=int(rng.integers(1e6)))
    Xs = pd.get_dummies(s[["flipper_length_mm","sex","species"]], drop_first=True, dtype=float)
    m = LinearRegression().fit(Xs,s["body_mass_g"])
    rows.append(dict(zip(Xs.columns, m.coef_)))
boot = pd.DataFrame(rows)

print("Bootstrap 95% CI:")
print(boot.quantile([0.025, 0.975]).T.round(2))
print("\nstatsmodels 95% CI:")
print(sm_fit.conf_int().round(2))

Bootstrap 95% CI:
                    0.025   0.975
flipper_length_mm   15.10   25.27
sex_male           462.05  608.10
species_Chinstrap -177.07    7.90
species_Gentoo     674.12  990.62

statsmodels 95% CI:
                               0        1
Intercept               -1412.48   680.84
C(sex)[T.male]            456.00   604.76
C(species)[T.Chinstrap]  -178.81     3.54
C(species)[T.Gentoo]      668.68  1003.84
flipper_length_mm          14.43    25.62


by bootstrapping we could resample the rows with replacement and refit the model as well as taake the middle 95% of the coefficents to get the intervals since there arent any standard errors.

In [5]:
#3a
X_full = pd.get_dummies(peng[["flipper_length_mm", "sex", "species"]], dtype=float)
print(list(X_full.columns))


['flipper_length_mm', 'sex_female', 'sex_male', 'species_Adelie', 'species_Chinstrap', 'species_Gentoo']


two for sex and three for species

In [6]:
#3b
lr_full = LinearRegression().fit(X_full, y)
print("R^2 (dummy):",round(lr.score(X, y), 6))
print("R^2 (one-hot):",round(lr_full.score(X_full, y), 6))
print("Same predictions:",np.allclose(lr.predict(X), lr_full.predict(X_full)))


R^2 (dummy): 0.866888
R^2 (one-hot): 0.866888
Same predictions: True


r2 is 0.866888 for both so theyre both the same

In [7]:
#3c

f = pd.Series(lr_full.coef_, index=X_full.columns)
d = pd.Series(lr.coef_, index=X.columns)
print("one-hot coefficients:"); print(f.round(4))
print("intercept:", round(lr_full.intercept_, 4))

print("\nDifferences in the one-hot model vs. the dummy-variable coefficients:")
print("Chinstrap - Adelie:", round(f["species_Chinstrap"]-f["species_Adelie"], 4), "dummy:", round(d["species_Chinstrap"], 4))
print("Gentoo - Adelie: ", round(f["species_Gentoo"] -f["species_Adelie"], 4), "dummy:", round(d["species_Gentoo"], 4))
print("male - female: ", round(f["sex_male"]- f["sex_female"], 4), "  dummy:", round(d["sex_male"], 4))

# The dummy intercept is the baseline (female Adelie)
print("\nintercept + sex_female + species_Adelie =",round(lr_full.intercept_ + f["sex_female"] + f["species_Adelie"], 4))
print("dummy intercept=", round(lr.intercept_, 4))

print("\nsex_female + sex_male is always 1:", (X_full["sex_female"]+X_full["sex_male"] == 1).all())
print("one-hot + intercept: columns =", X_full.shape[1]+1, " rank =", np.linalg.matrix_rank(np.column_stack([np.ones(len(X_full)), X_full])))
print("dummy + intercept:   columns =", X.shape[1] + 1, " rank =", np.linalg.matrix_rank(np.column_stack([np.ones(len(X)), X])))

one-hot coefficients:
flipper_length_mm     20.0249
sex_female          -265.1905
sex_male             265.1905
species_Adelie      -249.5418
species_Chinstrap   -337.1763
species_Gentoo       586.7182
dtype: float64
intercept: 148.9149

Differences in the one-hot model vs. the dummy-variable coefficients:
Chinstrap - Adelie: -87.6345 dummy: -87.6345
Gentoo - Adelie:  836.26 dummy: 836.26
male - female:  530.3811   dummy: 530.3811

intercept + sex_female + species_Adelie = -365.8174
dummy intercept= -365.8174

sex_female + sex_male is always 1: True
one-hot + intercept: columns = 7  rank = 5
dummy + intercept:   columns = 5  rank = 5


theyre not the same because sex_female = −265.19,sex_male =+265.19,  Adelie =−249.54, Chinstrap = −337.18, Gentoo = +586.72 and intercept = 148.91 with flipper coeff staying the same. the differences reproduce what the dummy variable gave us  incliding tge intercept. the dummy model gives us everything related to the dropped baseline and the other kust splits it across the level columns and intercept and they both describe the same model

3d
the one hot columns are collinear with the sexes both equalling 1 for every row. the matrix also gives only 5 when there are 7 which means that the otger ones just give the same predictions and sklearn picks the min norm solution and the fit and predictions stay the same

In [8]:
#4a

prep = ColumnTransformer([
    ("flipper", "passthrough", ["flipper_length_mm"]),
    ("cats", OneHotEncoder(drop="first"), ["sex", "species"]),
])
pipe = Pipeline([("prep", prep), ("model", LinearRegression())])
pipe.fit(peng[["flipper_length_mm", "sex", "species"]], y)

names = pipe.named_steps["prep"].get_feature_names_out()
print(pd.Series(pipe.named_steps["model"].coef_, index=names).round(4))
print("intercept:", round(pipe.named_steps["model"].intercept_, 4))
print("\nQ2 coefficients for comparison:"); print(pd.Series(lr.coef_, index=X.columns).round(4))
print("Q2 intercept:", round(lr.intercept_, 4))

flipper__flipper_length_mm     20.0249
cats__sex_male                530.3811
cats__species_Chinstrap       -87.6345
cats__species_Gentoo          836.2600
dtype: float64
intercept: -365.8174

Q2 coefficients for comparison:
flipper_length_mm     20.0249
sex_male             530.3811
species_Chinstrap    -87.6345
species_Gentoo       836.2600
dtype: float64
Q2 intercept: -365.8174


In [9]:
#4b

new = pd.DataFrame({"flipper_length_mm": [215], "sex": ["male"], "species": ["Gentoo"]})
print("pipeline prediction:", round(pipe.predict(new)[0], 4))
b = sm_fit.params
by_hand = b["Intercept"] + b["flipper_length_mm"] * 215 + b["C(sex)[T.male]"] + b["C(species)[T.Gentoo]"]
print(f"by hand: {b['Intercept']:.4f} + {b['flipper_length_mm']:.4f}*215 + {b['C(sex)[T.male]']:.4f} + {b['C(species)[T.Gentoo]']:.4f} = {by_hand:.4f}")

pipeline prediction: 5306.1805
by hand: -365.8174 + 20.0249*215 + 530.3811 + 836.2600 = 5306.1805


the prediction for the male gentoo is about 5306.18g which is similar to the pipeline output and because the encoder is already in the pipeline the same thing is applied to anything given to .predict().